# Create helper methods to:
 1. Load the CoDocGen model to generate documentation for a function
 2. Method that uses the CoDocModel model to generate documentation of the given code (as string)
 3. Load the the-stack dataset and extract only the C++ and python code from it.
 4. load the github/tree-sitter to create abstract syntx trees of given code and lanugage
 5. Create ASTs for the given code.

 Install all the necessary packages

In [ ]:
# Required for code doc generation
!pip install transformers torch accelerate  datasets

In [ ]:
# required for AST generator and comparator
!pip install tree-sitter tree-sitter-languages tree-sitter-cpp tree-sitter-python tree-sitter-cpp numpy

In [ ]:
# Upgrade torchao to a compatible version
!pip install --upgrade torchao

# CodeBERTScorer Package Installation

In [ ]:
!pip install sentence-transformers

In [ ]:
from google.colab import userdata
import os

# Force CPU to wait for GPU
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# Described debug output from GPU
os.environ["TORCH_USE_CUDA_DSA"] = "1"
# Force on single GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc #Added for memory cleanup
from dotenv import load_dotenv

# Debug flag - set to True to enable debug output
DEBUG_FLAG = False


# Try to retrieve the Hugging Face API key from Colab's secrets manager
HUGGING_FACE_KEY = userdata.get('HUGGING_FACE_KEY')

if HUGGING_FACE_KEY is None:
    print("WARNING: Hugging Face API key (HUGGING_FACE_KEY) not found in Colab secrets.")

    # Prompt user for input if not found in secrets
    HUGGING_FACE_KEY = input("Please enter your Hugging Face API Key: ")
    if not HUGGING_FACE_KEY:
        print("Hugging Face API Key was not provided. Some models might not load correctly.")
        import sys
        sys.exit()
    else:
        os.environ['HF_TOKEN'] = HUGGING_FACE_KEY
        print("Hugging Face API key received from input.")
else:
    # Set the environment variable for Hugging Face if found in secrets
    os.environ['HF_TOKEN'] = HUGGING_FACE_KEY
    print("Hugging Face API key loaded successfully from secrets.")

Hugging Face API key loaded successfully from secrets.


# All models will be singleton so create base metaclass for the singleton

In [ ]:
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):

        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(
                *args,
                **kwargs
            )

        return cls._instances[cls]

 # Singleton for the CodeGeneration

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, BatchEncoding
import torch

class QwenModelBase(metaclass=SingletonMeta):
    _initialized_qwen = False
    _model = None
    _tokenizer = None
    _model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"

    def __init__(self):
        if QwenModelBase._initialized_qwen:
            return
        QwenModelBase._initialized_qwen = True

        self.device = 'cuda' if torch.cuda.is_available() else 'cpu' # Determine device
        print(f"QwenModelBase using device: {self.device} for model {self._model_name}")
        self._load_qwen_model()

    def _load_qwen_model(self):
        """Loads the Qwen model and tokenizer."""
        if QwenModelBase._model is None:
            if self.device == 'cuda':
                # Use bfloat16 for Qwen (more stable than float16)
                try:
                    # Clear GPU cache before loading
                    torch.cuda.empty_cache()
                    QwenModelBase._model = AutoModelForCausalLM.from_pretrained(self._model_name,
                                                              torch_dtype=torch.bfloat16,
                                                              device_map="auto",
                                                              use_safetensors=True)
                except Exception as e:
                    print(f"bfloat16 loading failed, trying float32: {e}")
                    torch.cuda.empty_cache()
                    QwenModelBase._model = AutoModelForCausalLM.from_pretrained(self._model_name,
                                                              torch_dtype=torch.float32,
                                                              device_map="auto",
                                                              use_safetensors=True)
            else:
                QwenModelBase._model = AutoModelForCausalLM.from_pretrained(self._model_name)
                QwenModelBase._model.to(self.device)

            QwenModelBase._tokenizer = AutoTokenizer.from_pretrained(self._model_name)
            QwenModelBase._tokenizer.model_input_names = ['input_ids', 'attention_mask']

class CodeDocumentationGenerator(QwenModelBase):
    def __init__(self):
        """
        Constructor.

        IMPORTANT:
        Since SingletonMeta returns the same object every time,
        __init__ may be called multiple times.

        Therefore we guard against reinitialization.
        """

        if hasattr(self, "_initialized"):
            return

        super().__init__() # Initialize the QwenModelBase
        self._initialized = True
        print(f"CodeDocumentationGenerator using device: {self.device}")

        # Use the model and tokenizer from the base class
        self._model = QwenModelBase._model
        self._tokenizer = QwenModelBase._tokenizer


    def generate_documentation(self, code: str, max_length=512, num_return_sequences=1, prompt=None) -> list:
      """
        Generates documentation for a given code snippet using the CoDoCGen model.

      Args:
          code (str): The code for which to generate documentation.
          max_length (int): The maximum length of the generated documentation.
          num_return_sequences (int): The number of different documentation sequences to generate.

      Returns:
          list: A list of generated documentation strings.
      """
      if prompt is None:
        #prompt = f"generate good detailed documentation for what this software code does, do not include a pseudo code or example usage, just the intent of what the program should do, if it follows a design pattern, what actions to take under what conditions. The first line of the documentation should start with 'A software program in ProgLang, where ProgLan is the programming language of the program: {code}"
        prompt = f"generate a summary documentation for what this software code does, do not include a pseudo code or example usage, just the intent of what the program should do, if it follows a design pattern, and what actions to take under what conditions. Make the documentation generic about behavior of the code from outside the program or function. This documentation should allow writing the program in different programming language {code}"
      else:
        prompt = f"{prompt}"

      messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
      ]
      text = self._tokenizer.apply_chat_template( messages, tokenize=False, add_generation_prompt=True)
      model_inputs = self._tokenizer([text], return_tensors="pt").to(self.device)

      # Validate input token IDs are within valid range
      vocab_size = self._model.config.vocab_size
      input_ids = model_inputs["input_ids"]
      if torch.any(input_ids >= vocab_size) or torch.any(input_ids < 0):
          print(f"Warning: Qwen input contains invalid token IDs. Clipping to valid range [0, {vocab_size-1}]")
          model_inputs["input_ids"] = torch.clamp(input_ids, 0, vocab_size - 1)

      # Ensure input doesn't exceed model's max position embeddings
      max_pos = getattr(self._model.config, 'max_position_embeddings', 32768)
      input_len = model_inputs.input_ids.shape[1]
      if input_len >= max_pos:
          print(f"Warning: Input length {input_len} >= model max {max_pos}. Truncating.")
          truncated = {k: v[:, :max_pos-1] for k, v in model_inputs.items() if k in ['input_ids', 'attention_mask']}
          model_inputs = BatchEncoding(truncated)

      # Use safer generation parameters to avoid probability tensor errors
      generated_ids = None
      if True:
        with torch.no_grad():
            generated_ids = self._model.generate(
                **model_inputs,
                max_new_tokens=max_length,
                do_sample=False,  # Use greedy decoding for stability
                temperature=1.0,
            )

      if generated_ids is None:
          return "Error: Documentation generation failed"

      # Handle different output formats from generate()
      # The output can be a tensor, tuple, or ModelOutput object
      if isinstance(generated_ids, tuple):
          generated_ids = generated_ids[0]  # Take first element if tuple
      elif hasattr(generated_ids, 'input_ids'):
          # If it's a ModelOutput object
          generated_ids = generated_ids.input_ids

      # Get input length and slice generated_ids to remove input tokens
      input_len = model_inputs.input_ids.shape[1]
      generated_ids = generated_ids[:, input_len:]

      decoded = self._tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
      if not decoded:
          return ""
      response = decoded[0]
      return response

### Test the Code Documentation Generator

In [ ]:
if 'flag_demo_code_generation' in globals() and flag_demo_code_generation:
  generator = CodeDocumentationGenerator()

In [ ]:
if 'flag_demo_code_generation' in globals() and flag_demo_code_generation:
  print("\nGenerated Documentation 2:")
  documentation = generator.generate_documentation(sample_code)
  print(documentation)

# Obsolete TestLoad and Filter Code Dataset

A function to load a `bigcode/the-stack` dataset and filter it to include only C++ and Python code. The `language` column has the type of laguage

In [ ]:
if 'flag_dataset_load_and_filtering' in globals() and flag_dataset_load_and_filtering:
  from datasets import load_dataset, interleave_datasets

  def load_and_filter_code_dataset(languages:list =None):
      """
      Loads a code dataset and filters it by specified languages.

      Args:
          dataset_name (str): The name of the dataset to load (e.g., "codeparrot/github-code").
          languages (list): A list of programming languages to filter by (e.g., ['C++', 'Python']).
                            If None, no language filtering is applied.

      Returns:
          datasets.Dataset: The filtered dataset.
      """
      dataset_name ="bigcode/the-stack-dedup"
      languages_and_dataset = [
                                {
                                    'language':'python',
                                    'dataset':None
                                },
                                {
                                    'language':'cpp',
                                    'dataset':None
                                }
                              ]

      print(f"Loading dataset: {dataset_name}")
      train_dataset = None

      for i in range(len(languages_and_dataset)):
        language = languages_and_dataset[i]['language']
        print(f"Fetching dataset for '{language}' language")
        # 2. Merge them into one combined stream
        # probabilities=[0.5, 0.5] mixes them evenly (1 Python, 1 C++, 1 Python...)
        languages_and_dataset[i]['dataset'] = load_dataset(dataset_name,
                                        data_dir = f"data/{language}",
                                        split="train",
                                        streaming=True,
                                          token=True)


      return languages_and_dataset

### Obsolete Test: Loading and Filtering Code

Use the `load_and_filter_code_dataset` function to get only C++ and Python code from the `codeparrot/github-code` dataset.

In [ ]:
if 'flag_dataset_load_and_filtering' in globals() and flag_dataset_load_and_filtering:
  try:
      cpp_python_dataset = load_and_filter_code_dataset()

      print(next(iter(cpp_python_dataset[0]['dataset'])))
      print(next(iter(cpp_python_dataset[1]['dataset'])))
      # print("\nFirst example from filtered dataset (showing language and a snippet of code):")
      # first_example_filtered = next(iter(cpp_python_dataset))
      # print(f"---\nLanguage: {first_example_filtered.get('lang', 'N/A')}\nCode Snippet: {first_example_filtered.get('content', 'N/A')[:200]}...")

      # The original intent was to get 5 examples, but for streaming datasets,
      # directly indexing or taking len() can be problematic. Iterating explicitly is safer.
      # Let's just confirm the first example for now.

  except RuntimeError as e:
      if "Dataset scripts are no longer supported" in str(e):
          print(f"\nError loading dataset: {e}")
          print("\nIt appears the `codeparrot/github-code` dataset cannot be loaded directly via script anymore.")
          print("To fix this, please modify the `load_and_filter_code_dataset` function in cell `CnarbxKBNomR`.")
          print("You might need to specify a `config_name` (e.g., 'all' or 'code_x_m') and potentially use `streaming=True` if the dataset is very large, like so:")
          print("    `dataset = load_dataset(dataset_name, 'all', split=\"train\", streaming=True)`")
          print("Alternatively, you might need to find a different version of the dataset or a more compatible dataset for demonstration purposes.")
      else:
        raise e

In [ ]:
if 'flag_dataset_load_and_filtering' in globals() and flag_dataset_load_and_filtering:
  print("\nFirst example from cpp_python_dataset (using next(iter())):\n")
  first_example = next(iter(cpp_python_dataset))
  print(first_example)

# Singleton for AST Generation and Comparison

The `AST` class leverages `tree-sitter` to generate Abstract Syntax Trees (ASTs) from code snippets and provides a method to compare two ASTs. It is implemented as a singleton to ensure a single instance manages the language parsers and potentially future comparison models.

For generating the AST the input code can be a string or bytes. If string then convert to bytes before processing with tree sitter.


In [ ]:
import io
from contextlib import redirect_stdout
from typing import Union
from tree_sitter import Tree
import numpy as np

class AST(metaclass=SingletonMeta):
    def __init__(self):
        """
        Constructor for the AST singleton class.
        Initializes AST generator and comparator.
        Guards against reinitialization.
        """
        if hasattr(self, "_initialized"):
            return

        self._initialized = True
        self._load_ast_generator()

    def _load_ast_generator(self):
        """
        Loading AST generation models/parsers.
        """
        from tree_sitter import Language, Parser
        import tree_sitter_cpp as tscpp
        import tree_sitter_python as tspy

        # Load the language dynamically using tree-sitter-languages.
        # This function directly returns a tree_sitter.Language object.
        self._ast_cpp_parser = Parser()
        self._ast_cpp_parser.language = Language(tscpp.language())
        self._ast_python_parser = Parser()
        self._ast_python_parser.language = Language(tspy.language())


    def generate_ast(self, code: Union[str, bytes], language_name: str) -> Tree:
        """
          Generates an Abstract Syntax Tree (AST) for a given code snippet
          using tree-sitter for the specified language.

          Args:
          code (Union[str, bytes]): The code for which to generate the AST. Can be a string or bytestring.
          language_name (str): The name of the programming language (e.g., 'python', 'cpp').

        Returns:
        A tuple where:
        first element: tree_sitter.Tree: The generated AST.
        second element: string reresentation of the AST
        """

        # Check if the language is supported
        if language_name.lower() not in ['cpp', 'c++', 'python']:
          raise ValueError(f"Supported programming languages are 'C++', 'Python'. '{language_name}' is not supported!")

        # Encode string to bytes if necessary
        if isinstance(code, str):
          code = code.encode('utf-8')

        # Load the language dynamically using tree-sitter-languages.
        # This function directly returns a tree_sitter.Language object.
        ast_tree = None
        if language_name.lower() in ['cpp', 'c++']:
          ast_tree = self._ast_cpp_parser.parse(code)

        elif language_name.lower() == 'python':
          ast_tree = self._ast_python_parser.parse(code)

        else:
          raise ValueError(f"Runtime environment error for language {language_name}")

        return ast_tree

    def compare_ast(self, ast1: Tree, ast2: Tree) -> float:
        """
        Do not compare the raw Tree-Sitter trees, because
        'def add(a,b):' and 'def add( a , b):' althought same have different ASTs.
        instead normalize the representation.
        So (a quick search gave this):
        Step 1:  Keep node types, structure and remove puncutation and raw tokens.
        Step 2: Convet tree to ordered node list or S-expression
        Step 3: Compare using Jaccard similarity on AST node sequences.
                Jaccard Simnilrity = (Intersection / Union)
        Compares two ASTs (ASTs using structural node similarity) and returns a similarity score.
        Args:
            ast1 (Tree): The first AST tree_sitter.Tree object.
            ast2 (Tree): The second AST tree_sitter.Tree object.

        Returns:
            float: A similarity score between 0.0 and 1.0, where 1.0 means identical.
        """
         # Validate input ASTs
        if ast1 is None or ast2 is None:
            return 0.0
        if ast1.root_node is None or ast2.root_node is None:
            return 0.0

        def extract_ast_nodes(node):
          """
            Extracts only structural node types from Tree-sitter AST.
            This removes syntax noise like brackets, commas, etc.
          """
          nodes = []

          def dfs(n):
            # keep only meaningful AST node types
            if n.child_count > 0:
                nodes.append(n.type)

            for child in n.children:
                dfs(child)

          dfs(node)
          return nodes

        try:
            nodes1 = extract_ast_nodes(ast1.root_node)
            nodes2 = extract_ast_nodes(ast2.root_node)
        except (AttributeError, TypeError) as e:
            return 0.0

        set1 = set(nodes1)
        set2 = set(nodes2)

        intersection = len(set1 & set2)
        union = len(set1 | set2)

        if union == 0:
          return 0.0

        return intersection / union

    def to_str(self, tree: Tree)->str:
      '''
      Generate string format for the tree
      '''
      f = io.StringIO()
      with redirect_stdout(f):
          self.print_ast_tree(tree)
      return f.getvalue()

    def print_ast_tree(self, tree: Tree, indent=0):
      """
      Helper function to print AST nodes recursively with indentation.
      """
      def _print_node_recursive(node, current_indent):
          print(f"{_get_indent_string(current_indent)}{node.type} [start={node.start_point}, end={node.end_point}]")
          for child in node.children:
              _print_node_recursive(child, current_indent + 1)

      def _get_indent_string(current_indent):
          return '  ' * current_indent

      _print_node_recursive(tree.root_node, indent)


# GraphCodeBERTScorer for Semantic Code Similarity

The `GraphCodeBERTScorer` class is a singleton that uses a pre-trained `GraphCodeBERT` model (via `sentence-transformers`) to generate semantic embeddings for code snippets. It then calculates the cosine similarity between these embeddings to provide a semantic similarity score between two pieces of code. This is useful for tasks where AST comparison might miss semantic equivalence due to structural differences.

In [ ]:
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine
import numpy as np
import torch # Import torch for float16

class GraphCodeBERTScorer(metaclass=SingletonMeta):
    def __init__(self):
        """
        Constructor for the GraphCodeBERTScorer singleton class.
        Initializes the GraphCodeBERT model for semantic code similarity scoring.
        Guards against reinitialization.
        """
        if hasattr(self, "_initialized"):
            return

        self._initialized = True
        print("Initializing GraphCodeBERTScorer...")
        self._load_model()
        print("GraphCodeBERTScorer initialized successfully.")

    def _load_model(self):
        """
        Loads the pre-trained GraphCodeBERT model for generating code embeddings.
        """
        # Determine device dynamically
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Using device: {self.device}")

        # Using GraphCodeBERT-base from Hugging Face via sentence-transformers
        self.model = SentenceTransformer('microsoft/graphcodebert-base')
        # Move model to selected device and convert to float16 if on GPU for reduced memory usage
        if self.device == 'cuda':
            self.model.to(self.device, dtype=torch.float16)
        else:
            self.model.to(self.device)

    def score(self, code_1: str, code_2: str) -> float:
        """
        Calculates the semantic similarity score between two code snippets
        using GraphCodeBERT embeddings and cosine similarity.

        Args:
            code_1 (str): The first code snippet.
            code_2 (str): The second code snippet.

        Returns:
            float: A similarity score between 0.0 and 1.0, where 1.0 means identical semantics.
        """
        if not isinstance(code_1, str) or not isinstance(code_2, str):
            raise ValueError("Both code snippets must be strings.")

        # Generate GraphCodeBERT embeddings for both code snippets
        # Ensure encoding happens on the correct device
        embeddings = self.model.encode([code_1, code_2], convert_to_tensor=True, device=self.device)

        # Calculate cosine similarity between the embeddings
        # Cosine distance is 1 - cosine similarity, so similarity = 1 - distance
        similarity_score = 1 - cosine(embeddings[0].cpu().numpy(), embeddings[1].cpu().numpy())

        return float(similarity_score)


# LLM Judge
Use the "Qwen/Qwen2.5-Coder-7B-Instruct" to act as judge to say whether the code generated is as per the documentation or not.

In [ ]:
import json # Import json to parse the model's output
from typing import Optional # Keep Optional import

class LLMJudge(QwenModelBase):
    def __init__(self, device: str = None): # Added device parameter
        if hasattr(self, "_initialized"):
            return
        super().__init__() # Initialize the QwenModelBase
        self._initialized = True
        # If a device was explicitly passed, it would be 'device'. Otherwise, self.device is from QwenModelBase.
        self.device = device if device is not None else self.device
        # Since QwenModelBase is a singleton and sets self.device, we'll rely on that for consistency.
        print(f"LLMJudge using device: {self.device}")

        self._model = QwenModelBase._model
        self._tokenizer = QwenModelBase._tokenizer
        self._print_once = False # Initialize the flag for printing JSON once

    def qwen_code_judge(self, documentation: str, generated_code: str, reference_code: Optional[str] = None) -> float:
        """
        Uses the Qwen model to judge the quality of generated code based on documentation and an optional reference code.

        Args:
            documentation (str): The original documentation for the code.
            generated_code (str): The code generated by another model.
            reference_code (Optional[str]): The ground truth or reference code for comparison.

        Returns:
            float: The score from the Qwen model's judgment.
        """
        if reference_code:
            prompt = (
                f"Given the following documentation:\n\n{documentation}\n\n"
                f"And a generated code snippet:\n\n```\n{generated_code}\n```\n\n"
                f"Compare it against the reference code:\n\n```\n{reference_code}\n```\n\n"
                f"Evaluate the generated code. Respond ONLY with valid JSON:\
                  {{\
                    'score': <0-1>,\
                    'compiles': <true/false>,\
                    'logic_correct': <true/false>,\
                    'handles_edge_cases': <true/false>,\
                    'issues': ['issue1', 'issue2'],\
                    'verdict': 'correct' | 'incorrect' | 'partially_correct',\
                    'explanation': '<one sentence>'\
                  }}"

            )
        else:
            prompt = (
                f"Given the following documentation:\n\n{documentation}\n\n"
                f"And a generated code snippet:\n\n```\n{generated_code}\n```\n\n"
                f"Evaluate the generated code. Respond ONLY with valid JSON:\
                  {{\
                    'score': <0-1>,\
                    'compiles': <true/false>,\
                    'logic_correct': <true/false>,\
                    'handles_edge_cases': <true/false>,\
                    'issues': ['issue1', 'issue2'],\
                    'verdict': 'correct' | 'incorrect' | 'partially_correct',\
                    'explanation': '<one sentence>'\
                  }}"
            )

        messages = [
            {"role": "system", "content": "You are an expert C++ code reviewer."},
            {"role": "user", "content": prompt}
        ]
        text = self._tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = self._tokenizer([text], return_tensors="pt").to(self.device)

        with torch.no_grad():
          generated_ids = self._model.generate(**model_inputs, max_new_tokens=512, temperature=0.1)

        # Get input length and slice generated_ids to remove input tokens
        input_len = model_inputs.input_ids.shape[1]
        generated_ids = generated_ids[:, input_len:]

        decoded = self._tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        if not decoded:
            print("Warning: LLM Judge returned empty response")
            return 0.0
        response_str = decoded[0]

        try:
            response_json = json.loads(response_str)
            score = float(response_json.get("score", 0.0)) # Extract the score

            # Conditional printing of the full JSON response
            if not self._print_once:
                print("\n--- LLM Judge Response (printed once) ---")
                print(json.dumps(response_json, indent=2))
                print("-------------------------------------------")
                self._print_once = True # Set flag to true after printing

            return score
        except json.JSONDecodeError:
            print(f"Warning: Could not decode JSON response from LLM Judge:\n{response_str}")
            return 0.0 # Return 0.0 if JSON is invalid
        except ValueError:
            print(f"Warning: 'score' field not a valid number in JSON response:\n{response_str}")
            return 0.0 # Return 0.0 if score is not a valid number

# Singleton for Filtered and Enriched Code Dataset

This `FilteredDataset` class acts as a singleton to efficiently manage access to and processing of the `bigcode/the-stack-dedup` dataset. It specifically filters for C++ and Python code, and for each valid code snippet, it generates detailed documentation using the `CodeDocumentationGenerator` and constructs an Abstract Syntax Tree (AST) using the `AST` processor. This enrichment happens dynamically as you iterate through the dataset, providing a stream of ready-to-use data for tasks like code generation or analysis.

In [ ]:
from datasets import load_dataset
from typing import Iterator, Dict, Any, Union
from tree_sitter import Tree # Import Tree for type hinting in AST class, though not directly used in FilteredDataset

class FilteredDataset(metaclass=SingletonMeta):
    _initialized = False # Class-level flag for singleton initialization

    def __init__(self):
        """
        Constructor for the FilteredDataset singleton class.
        Initializes dataset loading and processing components.
        """
        if FilteredDataset._initialized:
            print("FilteredDataset already initialized. Returning existing instance.")
            return
        FilteredDataset._initialized = True

        print("Initializing FilteredDataset singleton...")
        self._local_documentation_cache: Dict[str, str] = {} # Cache for local documentation updates
        self._local_scores_cache: Dict[str, Dict[str, float]] = {} # Cache for storing scores
        self._python_dataset_stream = None
        self._cpp_dataset_stream = None
        self._python_iter = None # Internal iterator for the combined stream
        self._cpp_iter = None # Internal iterator for the combined stream
        self._current_dataset_selector = 0 # 0 for Python, 1 for C++self._cached_dataset = None # Local cached dataset for faster iteration
        self._use_cached = False # Flag to use cached dataset instead of streaming
        self._load_datasets() # Initial loading of the datasets
        self._num_samples=0
        print("FilteredDataset initialized successfully.")

    def load_cached_dataset(self, cache_dir: str = "./cached_data", num_samples: int = 100):
        """
        Load a previously cached dataset from disk for faster iteration.

        Args:
            cache_dir (str): Directory where the cached dataset is stored.
            num_samples (int): Number of samples the cached dataset contains.
        """
        import os
        from datasets import Dataset

        cached_file_path = os.path.join(cache_dir, f"filtered_subset_{num_samples}")
        if os.path.exists(cached_file_path):
            print(f"Loading cached dataset from {cached_file_path}...")
            self._cached_dataset = Dataset.load_from_disk(cached_file_path)
            self._use_cached = True
            print(f"Cached dataset loaded with {len(self._cached_dataset)} samples.")
        else:
            print(f"No cached dataset found at {cached_file_path}")
            self._use_cached = False

    def _convert_max_stars_count(self, example: Dict[str, Any]) -> Dict[str, Any]:
        """
        Helper function to convert 'max_stars_count' to float64, handling missing or invalid values.
        """
        if 'max_stars_count' in example and example['max_stars_count'] is not None:
            try:
                example['max_stars_count'] = float(example['max_stars_count'])
            except (ValueError, TypeError):
                example['max_stars_count'] = None # Assign None for conversion errors
        return example

    def _load_datasets(self):
        """
        Helper method to load the datasets separately and initialize their iterators.
        This also applies the 'max_stars_count' conversion.
        """
        print("Loading Python dataset...")
        python_raw_stream = load_dataset("bigcode/the-stack-dedup", data_dir="data/python", split="train", streaming=True, token=True)
        self._python_dataset_stream = python_raw_stream.map(self._convert_max_stars_count)

        print("Loading C++ dataset...")
        cpp_raw_stream = load_dataset("bigcode/the-stack-dedup", data_dir="data/cpp", split="train", streaming=True, token=True)
        self._cpp_dataset_stream = cpp_raw_stream.map(self._convert_max_stars_count)

        # Initialize iterators for the main __iter__ method
        self._python_iter = iter(self._python_dataset_stream)
        self._cpp_iter = iter(self._cpp_dataset_stream)
        self._current_dataset_selector = 0 # Reset selector for alternating iteration
        python_len = len(self._python_dataset_stream) if hasattr(self._python_dataset_stream, '__len__') else "streaming"
        cpp_len = len(self._cpp_dataset_stream) if hasattr(self._cpp_dataset_stream, '__len__') else "streaming"
        print(f"Python dataset length: {python_len}")
        print(f"C++ dataset length: {cpp_len}")
        if python_len != "streaming" and cpp_len != "streaming":
            print(f"Total dataset length: {python_len + cpp_len}")

    def reset_iterator(self):
        """
        Resets the dataset iterator to the beginning, allowing re-iteration from the start.
        Automatically attempts to load a cached dataset if available, otherwise streams from HuggingFace.
        """
        print("Resetting dataset streams and iterators...")
        self._num_samples = 0

        # Try to auto-load cached dataset if exists
        if not self._use_cached or self._cached_dataset is None:
            self._try_load_cached_dataset()

        if self._use_cached and self._cached_dataset is not None:
            # Use cached dataset for faster iteration
            print("Using cached dataset for iteration...")
        else:
            # Fall back to streaming from HuggingFace
            self._load_datasets() # Re-call to reload the streams and create new iterators

    def _try_load_cached_dataset(self):
        """Try to load a cached dataset from the default location."""
        import os
        from datasets import Dataset

        default_cache_dir = "./cached_data"
        if not os.path.exists(default_cache_dir):
            return

        # Look for cached datasets
        for filename in os.listdir(default_cache_dir):
            if filename.startswith("filtered_subset_"):
                cached_file_path = os.path.join(default_cache_dir, filename)
                try:
                    print(f"Auto-loading cached dataset from {cached_file_path}...")
                    self._cached_dataset = Dataset.load_from_disk(cached_file_path)
                    self._use_cached = True
                    print(f"Cached dataset loaded with {len(self._cached_dataset)} samples.")
                    return
                except Exception as e:
                    print(f"Could not load cached dataset: {e}")
                    continue

    def _enrich_record(self, example: Dict[str, Any]) -> Dict[str, Any]:
        """
        Helper method to apply common enrichment logic to a single record.
        This includes adding language, documentation (from cache), score placeholders,
        and a flag indicating if the code is processable.
        """
        code_content = example.get('content')
        original_language = example.get('lang')
        hexsha = example.get('hexsha')

        example['language'] = original_language

        # Prioritize local documentation cache
        example['documentation'] = self._local_documentation_cache.get(hexsha, "")

        # Initialize score fields
        example['ast_score'] = self._local_scores_cache.get(hexsha, {}).get('ast_score', None)
        example['graphcodebert_score'] = self._local_scores_cache.get(hexsha, {}).get('graphcodebert_score', None)
        example['average_score'] = self._local_scores_cache.get(hexsha, {}).get('average_score', None)

        # New fields for PL1->PL2 translation scores
        example['python_translation_ast_score'] = self._local_scores_cache.get(hexsha, {}).get('python_translation_ast_score', None)
        example['python_translation_gcb_score'] = self._local_scores_cache.get(hexsha, {}).get('python_translation_gcb_score', None)
        example['python_translation_average_score'] = self._local_scores_cache.get(hexsha, {}).get('python_translation_average_score', None)

        example['ast_tree'] = None # AST object itself is not serializable
        example['ast_string_representation'] = ""

        # Determine if the code is processable
        if code_content and original_language and isinstance(code_content, str): # and 50 < len(code_content) < 2000:
            example['is_processable_code'] = True
        else:
            example['is_processable_code'] = False
            print(f"Not processable code for {original_language} for code type {type(code_content)} and code {code_content}")
            if hexsha not in self._local_documentation_cache:
                example['documentation'] = "Skipped: Code content or language invalid/missing."
            example['ast_string_representation'] = "Skipped: Code content or language invalid/missing."
        return example

    def __iter__(self) -> Iterator[Dict[str, Any]]:
        """
        Iterates over the combined (Python and C++) streaming dataset by alternating.
        If a cached dataset is available, iterates over that instead.
        Enriches each record with language, documentation from local cache (if any),
        score placeholders, and a flag indicating if the code is processable.

        Yields:
            Dict[str, Any]: An enriched dictionary representing a single record from the dataset.
        """
         # If using cached dataset, iterate over that
        if self._use_cached and self._cached_dataset is not None:
            print("Starting iteration over cached dataset...")
            for example in self._cached_dataset:
                self._num_samples += 1
                if (self._num_samples % 100) == 0:
                    print(f"Gave {self._num_samples}")
                yield self._enrich_record(example)
            return

        print("Starting iteration over combined dataset (alternating Python and C++) via __iter__...")
        # The iterators _python_iter and _cpp_iter are managed by _load_datasets/reset_iterator
        # and maintain their state across calls to next() for a single __iter__ session.
        while self._python_iter is not None or self._cpp_iter is not None:
            example = None
            if self._current_dataset_selector == 0: # Try to get from Python dataset
                if self._python_iter:
                    try:
                        example = next(self._python_iter)
                        self._current_dataset_selector = 1 # Switch to C++ for next iteration
                    except StopIteration:
                        self._python_iter = None # Python stream exhausted
                        self._current_dataset_selector = 1 # Try C++ next if Python is exhausted
                else: # Python already exhausted, switch to C++
                    self._current_dataset_selector = 1

            if example is None and self._current_dataset_selector == 1: # Try to get from C++ dataset (either after Python or directly)
                if self._cpp_iter:
                    try:
                        example = next(self._cpp_iter)
                        self._current_dataset_selector = 0 # Switch to Python for next iteration
                    except StopIteration:
                        self._cpp_iter = None # C++ stream exhausted
                        self._current_dataset_selector = 0 # Try Python next if C++ is exhausted
                else: # C++ already exhausted, switch to Python
                    self._current_dataset_selector = 0

            if example is None: # Both streams might be exhausted or only one was active and now exhausted
                if self._python_iter is None and self._cpp_iter is None:
                    break # Both exhausted, stop iteration
                else:
                    # One stream exhausted, but the other might still have data.
                    # The selector should naturally point to the non-exhausted one.
                    # We just continue the loop to try fetching from the other stream.
                    continue

            self._num_samples +=1
            if DEBUG_FLAG and (self._num_samples%100) ==0:
                print(f"\tDEBUG - Iterator Gave {self._num_samples}")

            yield self._enrich_record(example)

    def get_python_stream_iterator(self) -> Iterator[Dict[str, Any]]:
        """
        Returns a fresh iterator for the Python dataset stream, applying enrichment to each record.
        This iterator is independent of the main alternating iterator.
        """
        if self._python_dataset_stream is None:
            # If streams are not loaded, load them
            self._load_datasets()
        # Return a new iterator each time this method is called to allow for independent iteration
        return (self._enrich_record(record) for record in iter(self._python_dataset_stream))

    def get_cpp_stream_iterator(self) -> Iterator[Dict[str, Any]]:
        """
        Returns a fresh iterator for the C++ dataset stream, applying enrichment to each record.
        This iterator is independent of the main alternating iterator.
        """
        if self._cpp_dataset_stream is None:
            # If streams are not loaded, load them
            self._load_datasets()
        # Return a new iterator each time this method is called to allow for independent iteration
        return (self._enrich_record(record) for record in iter(self._cpp_dataset_stream))

    def update_documentation(self, record_identifier: str, new_documentation: str):
        """
        Updates the documentation for a specific record locally within the dataset instance.
        This updated documentation will be returned by subsequent iterations when that
        record's `record_identifier` (hexsha) is encountered.

        Args:
            record_identifier (str): The unique identifier (e.g., 'hexsha') of the record to update.
            new_documentation (str): The new documentation string to associate with the record.
        """
        if not isinstance(record_identifier, str):
            raise TypeError("record_identifier must be a string (e.g., 'hexsha').")
        self._local_documentation_cache[record_identifier] = new_documentation
        print(f"\tDocumentation for record '{record_identifier}' updated locally.")

    def update_scores(self, record_identifier: str, ast_score: float, graphcodebert_score: float, average_score: float,
                      python_translation_ast_score: Optional[float] = None,
                      python_translation_gcb_score: Optional[float] = None,
                      python_translation_average_score: Optional[float] = None):
        """
        Updates the scores for a specific record locally within the dataset instance.

        Args:
            record_identifier (str): The unique identifier (e.g., 'hexsha') of the record to update.
            ast_score (float): The AST similarity score for NL->PL.
            graphcodebert_score (float): The GraphCodeBERT semantic similarity score for NL->PL.
            average_score (float): The average of AST and GraphCodeBERT scores for NL->PL.
            python_translation_ast_score (Optional[float]): The AST similarity score for NL->C++->Python translation.
            python_translation_gcb_score (Optional[float]): The GraphCodeBERT semantic similarity score for NL->C++->Python translation.
            python_translation_average_score (Optional[float]): The average score for NL->C++->Python translation.
        """
        if not isinstance(record_identifier, str):
            raise TypeError("record_identifier must be a string (e.g., 'hexsha').")

        # Fetch existing scores or initialize if not present
        current_scores = self._local_scores_cache.get(record_identifier, {})

        current_scores.update({
            'ast_score': ast_score,
            'graphcodebert_score': graphcodebert_score,
            'average_score': average_score
        })

        if python_translation_ast_score is not None:
            current_scores['python_translation_ast_score'] = python_translation_ast_score
        if python_translation_gcb_score is not None:
            current_scores['python_translation_gcb_score'] = python_translation_gcb_score
        if python_translation_average_score is not None:
            current_scores['python_translation_average_score'] = python_translation_average_score

        self._local_scores_cache[record_identifier] = current_scores
        print(f"\tScores for record '{record_identifier}' updated locally")
        print(f"\t\tNL->PL Avg={average_score:.4f}")
        print(f"\t\tNL->C++->Py Avg={python_translation_average_score:.4f}" if python_translation_average_score is not None else "")


# Efficient Filtering and Caching of Streaming Data

For large streaming datasets, repeatedly iterating and filtering can still be slow. Use `datasets` library features for more efficient filtering and to cache a *subset* of your data if you need to iterate over the same items multiple times:

1.  **Direct Filtering with `IterableDataset.filter()`**: You can apply a `filter()` method directly to `IterableDataset` objects. This is efficient as it processes data on-the-fly without loading the entire stream into memory.

2.  **Materializing a Filtered Subset**: If you need to repeatedly access a *specific, small subset* of the filtered data, it's beneficial to materialize it into a non-streaming `Dataset` and save it to disk. This avoids re-fetching and re-processing from the streaming source every time you iterate or reset.

## Materializing and Enriching the Full Dataset

To persist the entire dataset with generated documentation and ASTs, and to incorporate any local documentation updates, we'll use a function that iterates through the streaming `FilteredDataset`, performs the enrichment, and saves the result as a non-streaming `datasets.Dataset` to disk. This approach is memory-efficient for large datasets by leveraging `datasets.Dataset.from_generator`.

In [ ]:
import os
from datasets import Dataset, IterableDataset
from tqdm import tqdm
from functools import partial
from typing import Dict, Any, Optional

def _enrich_single_record(
    example: Dict[str, Any],
    local_doc_cache: Optional[Dict[str, str]] = None # Pass the cache for overrides
) -> Dict[str, Any]:
    """
    Helper to prepare a single record for materialization. It applies local documentation overrides if any.
    Documentation and AST generation are assumed to be done by the user before materialization
    or are handled by the FilteredDataset's __iter__ for initial defaults.
    """
    hexsha = example.get('hexsha')

    # Prioritize local cache for documentation if available
    if local_doc_cache and hexsha in local_doc_cache:
        example['documentation'] = local_doc_cache[hexsha]
    # else, example['documentation'] already contains the default from FilteredDataset.__iter__

    # ast_tree is non-serializable and should always be removed before saving
    example.pop('ast_tree', None)

    # ast_string_representation should already be set by FilteredDataset.__iter__
    # (either empty or 'Skipped...') and is not generated here, as requested.

    return example


def materialize_and_enrich_dataset(
    filtered_dataset_instance: FilteredDataset,
    output_cache_path: str = "./enriched_full_dataset",
    force_reprocess: bool = False,
    num_samples_to_process: Optional[int] = None # Limit for testing or smaller datasets
) -> Dataset:
    """
    Materializes the entire streaming dataset (or a subset), ensuring local documentation overrides
    are applied, and caches the dataset to disk using Dataset.from_generator for memory efficiency.
    Documentation and AST generation are externalized to the user.

    Args:
        filtered_dataset_instance (FilteredDataset): An instance of the FilteredDataset singleton.
        output_cache_path (str): Directory where the enriched dataset will be saved/loaded.
        force_reprocess (bool): If True, re-process and overwrite existing cache.
        num_samples_to_process (int, optional): If provided, processes only this many samples.
                                                Useful for testing with large datasets.

    Returns:
        datasets.Dataset: The fully materialized and enriched dataset.
    """
    os.makedirs(output_cache_path, exist_ok=True)

    if os.path.exists(os.path.join(output_cache_path, 'state.json')) and not force_reprocess:
        print(f"Loading existing enriched dataset from {output_cache_path}...")
        return Dataset.load_from_disk(output_cache_path)

    print(f"Materializing and enriching dataset to {output_cache_path}...")

    local_doc_cache = filtered_dataset_instance._local_documentation_cache # Get the current local overrides

    enrich_func = partial(
        _enrich_single_record,
        local_doc_cache=local_doc_cache
    )

    def generator_function():
        filtered_dataset_instance.reset_iterator() # Start from clean stream
        processed_count = 0
        # Use tqdm to show progress for large operations
        for example in tqdm(filtered_dataset_instance, desc="Materializing records"): # Assuming filtered_dataset_instance itself is iterable
            if num_samples_to_process is not None and processed_count >= num_samples_to_process:
                break

            enriched_example = enrich_func(example)
            yield enriched_example
            processed_count += 1

    print("Creating enriched dataset from generator (this may take a long time for full dataset)...")
    materialized_dataset = Dataset.from_generator(generator_function)

    print(f"Saving enriched dataset to {output_cache_path}...")
    materialized_dataset.save_to_disk(output_cache_path)
    print("Enriched dataset saved successfully.")

    return materialized_dataset


In [ ]:
import os
from datasets import Dataset

def create_and_cache_filtered_subset(filtered_dataset_instance: FilteredDataset, num_samples_to_cache: int, cache_dir: str = "./cached_data") -> Dataset:
    """
    Collects a specified number of filtered and processable samples from the streaming dataset,
    materializes them into a non-streaming Dataset, and saves them to disk.

    Args:
        filtered_dataset_instance (FilteredDataset): An instance of the FilteredDataset singleton.
        num_samples_to_cache (int): The number of valid samples to collect and cache.
        cache_dir (str): Directory to save the cached dataset.

    Returns:
        datasets.Dataset: The materialized and cached dataset.
    """
    os.makedirs(cache_dir, exist_ok=True)
    cached_file_path = os.path.join(cache_dir, f"filtered_subset_{num_samples_to_cache}")

    if os.path.exists(cached_file_path):
        print(f"Loading cached dataset from {cached_file_path}...")
        return Dataset.load_from_disk(cached_file_path)

    print(f"Generating and caching {num_samples_to_cache} samples...")
    raw_examples = []
    count = 0
    for example in filtered_dataset_instance:
        if example['is_processable_code']:
            # Optionally, perform heavy processing here if you want to cache the processed results
            # For this example, we're just caching the raw processable examples.

            # To avoid excessive memory usage, we'll store only relevant keys
            # You can customize which keys to keep based on your needs
            clean_example = {
                'hexsha': example.get('hexsha'),
                'content': example.get('content'),
                'lang': example.get('lang'),
                'language': example.get('language'),
                'is_processable_code': example.get('is_processable_code', True)
            }
            raw_examples.append(clean_example)
            count += 1
            if count >= num_samples_to_cache:
                break

    if not raw_examples:
        raise ValueError("No processable samples found to create a cached dataset.")

    print(f"Collected {len(raw_examples)} samples. Converting to Dataset...")
    cached_dataset = Dataset.from_list(raw_examples)
    cached_dataset.save_to_disk(cached_file_path)
    print(f"Cached dataset saved to {cached_file_path}")
    return cached_dataset


Initializing FilteredDataset singleton...
Loading Python dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/144 [00:00<?, ?it/s]

Loading C++ dataset...


Resolving data files:   0%|          | 0/110 [00:00<?, ?it/s]

FilteredDataset initialized successfully.
Loading cached dataset from ./cached_data/filtered_subset_100...

Successfully created/loaded a cached dataset with 100 samples.
First 2 samples from cached dataset:
{'hexsha': 'd99a1e98eccb58cbc0c0cef6e9e6702f33461b0e', 'content': 'from rest_framework_gis import serializers\nfrom rest_framework import serializers as s\n\nfrom .models import (\n    Artificialisee2015to2018,\n    Artificielle2018,\n    CommunesSybarval,\n    CouvertureSol,\n    EnveloppeUrbaine2018,\n    Ocsge,\n    Renaturee2018to2015,\n    Sybarval,\n    Voirie2018,\n    ZonesBaties2018,\n    UsageSol,\n)\n\n\ndef get_label(code="", label=""):\n    if code is None:\n        code = "-"\n    if label is None:\n        label = "inconnu"\n    return f"{code} {label[:30]}"\n\n\nclass Artificialisee2015to2018Serializer(serializers.GeoFeatureModelSerializer):\n    usage_2015 = s.SerializerMethodField()\n    usage_2018 = s.SerializerMethodField()\n    couverture_2015 = s.SerializerMet

# Baseline
use the codegen-350m-multi model, and for each record in the dataset:
1. generte the documentation
2. give the model the documentation and ask it generate the code from the programming language of the record
3. create AST of the ground truth in the record and the generated program.
4. Compare the AST to generate the similarity score
5. Compare the generate code using CoderBERTScore

### Baseline Data Generation and Scoring

The `BaselineData` class orchestrates the generation of documentation, code, and subsequent scoring using AST similarity and GraphCodeBERT semantic similarity. It iterates through records, attempts multiple code generations from documentation, and stores the best-performing results for each record.

In [ ]:
class BaselineData(metaclass=SingletonMeta):
    _initialized = False # Class-level flag for singleton initialization
    _preferred_device = None  # Class-level device preference (can be set externally)
    _num_samples = 100  # Default number of samples

    def __init__(self, num_samples: int = 100):
        """
        Constructor for the BaselineData singleton class.
        Initializes all necessary components for baseline evaluation.

        Args:
            num_samples (int): Number of samples to use from FilteredDataset.
                              If cache exists for this count, it will be loaded.
                              Otherwise, streaming will occur and cache will be created.
        """
        if BaselineData._initialized:
            print("BaselineData already initialized. Returning existing instance.")
            return
        BaselineData._initialized = True
        BaselineData._num_samples = num_samples

        # Use preferred device if set, otherwise auto-detect
        if BaselineData._preferred_device:
            self.device = BaselineData._preferred_device
        else:
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Initializing BaselineData singleton using device: {self.device}...")

        # Initialize dependencies internally as per user request to decouple FilteredDataset
        self._documentation_generator = CodeDocumentationGenerator()
        self._ast_processor = AST()
        self._graphcodebert_scorer = GraphCodeBERTScorer()
        self._filtered_dataset = FilteredDataset()
        self._llm_judge = LLMJudge(self.device) # Initialize the LLMJudge here, passing self.device

        # Create or load cached dataset for the specified number of samples
        self._cached_dataset = create_and_cache_filtered_subset(
            self._filtered_dataset, num_samples, "./cached_data"
        )

        # Load codegen model for code generation from documentation
        self._load_codegen_model()

        print("BaselineData initialized successfully.")

    def _load_codegen_model(self):
        """
        Loads the Salesforce/codegen-350M-multi model for code generation.
        """
        model_name_codegen = "Salesforce/codegen-350M-multi"
        print(f"Loading code generation model: {model_name_codegen} on {self.device}...")

        # Force GPU reset before loading
        if torch.cuda.is_available():
            try:
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
                torch.cuda.reset_peak_memory_stats()
            except RuntimeError:
                pass

        self._tokenizer_codegen = AutoTokenizer.from_pretrained(model_name_codegen)
        self._tokenizer_codegen.pad_token = self._tokenizer_codegen.eos_token
        # Ensure pad_token_id is explicitly set, common for causal models where EOS is used for padding
        if self._tokenizer_codegen.pad_token_id is None:
            self._tokenizer_codegen.pad_token_id = self._tokenizer_codegen.eos_token_id

        if self.device == 'cuda':
            # Use float32 for stability (float16 can cause index out of bounds issues)
            # Load model with device_map="auto" which handles GPU placement automatically
            # This avoids the .to() call that triggers CUDA index errors

            # Clear any corrupted cache first
            import shutil
            from pathlib import Path
            cache_dir = Path.home() / ".cache" / "huggingface" / "hub"
            codegen_cache = list(cache_dir.glob("*codegen*")) if cache_dir.exists() else []
            for item in codegen_cache:
                try:
                    if item.is_dir():
                        shutil.rmtree(item)
                    else:
                        item.unlink()
                    print(f"Cleared corrupted cache: {item}")
                except Exception as ex:
                    print(f"Could not clear cache {item}: {ex}")

            try:
                print(f"Loading model {model_name_codegen} to GPU...")
                self._model_codegen = AutoModelForCausalLM.from_pretrained(
                    model_name_codegen,
                    torch_dtype=torch.float32,
                    device_map="auto",
                    trust_remote_code=False
                )
                print(f"Model loaded successfully. Config: {self._model_codegen.config}")
                self._model_codegen.eval()
            except RuntimeError as e:
                error_msg = str(e)
                print(f"Model loading error: {error_msg[:500]}")
                raise
        else:
            self._model_codegen = AutoModelForCausalLM.from_pretrained(model_name_codegen)
            self._model_codegen.to(self.device)

        # When using Salesforce/codegen-350M-multi for translation tasks, this usually happens because of an unconfigured pad token or a context window overflow
        self._model_codegen.config.pad_token_id = self._model_codegen.config.eos_token_id
        print("Code generation model loaded.")

    def _generate_code_from_model(self, input_text: str, target_lang: str, is_py_to_cpp: bool = False) -> str:
        """
        Generates code using the base codegen model from a given input text (documentation or code).

        Args:
            input_text (str): The input text (documentation or code) to generate from.
            target_lang (str): The target programming language for generation (e.g., 'python', 'cpp').
            is_cpp_to_py (bool): If True, implies C++ to Python translation prompt format.

        Returns:
            str: The generated code.
        """
        if is_py_to_cpp:
            prompt = f"Convert the following Python code to C++:\n{input_text}\nC++ code:\n"
        else:
            prompt = f"Generate {target_lang} code based on the following documentation:\n{input_text}\n{target_lang} code:"

        # Store prompt for potential re-tokenization during retries
        original_prompt = prompt

        # Option 1 & 2: Reduced max_new_tokens and added memory cleanup
        max_new_tokens = 128  # Reduced from 256 to prevent CUDA errors

        # Get model's max position embeddings to prevent CUDA index out of bounds errors
        model_max_length = getattr(self._model_codegen.config, 'max_position_embeddings', 2048)
        n_positions = getattr(self._model_codegen.config, 'n_positions', model_max_length)
        if DEBUG_FLAG:
            print(f"DEBUG - Model config - max_position_embeddings: {model_max_length}, n_positions: {n_positions}")
            print(f"DEBUG - Model config - hidden_size: {self._model_codegen.config.hidden_size}")
        # Ensure total sequence (input + output) doesn't exceed model's max position embeddings
        # Use a very conservative limit to prevent position index out of bounds
        max_input_length = min(model_max_length - max_new_tokens - 100, 256)  # Extra safety margin

        inputs = self._tokenizer_codegen(prompt, truncation=True, return_tensors="pt", max_length=256, padding=True).to(self.device)
        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]

        # Ensure attention mask is valid (no NaN or inf)
        if torch.isnan(attention_mask).any() or torch.isinf(attention_mask).any():
            print("Warning: Attention mask contains NaN or inf values, replacing with ones")
            attention_mask = torch.where(torch.isnan(attention_mask) | torch.isinf(attention_mask), torch.ones_like(attention_mask), attention_mask)

        # Ensure attention mask shape matches input_ids shape
        if attention_mask.shape != input_ids.shape:
            print(f"Warning: Attention mask shape {attention_mask.shape} != input_ids shape {input_ids.shape}, recreating attention mask")
            attention_mask = torch.ones_like(input_ids)

        # Truncate input if it exceeds model's context window
        if input_ids.shape[1] > max_input_length:
            print(f"Warning: Input length {input_ids.shape[1]} exceeds model context window. Truncating to {max_input_length} tokens.")
            input_ids = input_ids[:, -max_input_length:]
            attention_mask = attention_mask[:, -max_input_length:]

        # Validate input token IDs are within valid range
        vocab_size = self._model_codegen.config.vocab_size
        if DEBUG_FLAG:
            print(f"DEBUG - Model vocab_size: {vocab_size}")

        # Check for NaN or inf values in input_ids
        if torch.isnan(input_ids).any() or torch.isinf(input_ids).any():
            print("Warning: Input contains NaN or inf values, replacing with zeros")
            input_ids = torch.where(torch.isnan(input_ids) | torch.isinf(input_ids), torch.zeros_like(input_ids), input_ids)

        # Validate and clip token IDs to valid range
        if torch.any(input_ids >= vocab_size) or torch.any(input_ids < 0):
            invalid_count = torch.sum((input_ids >= vocab_size) | (input_ids < 0)).item()
            print(f"Warning: Input contains {invalid_count} invalid token IDs. Clipping to valid range [0, {vocab_size-1}]")
            input_ids = torch.clamp(input_ids, 0, vocab_size - 1)

        # Ensure all token IDs are valid integers
        input_ids = input_ids.long()

        # Ensure position IDs don't exceed model's max position embeddings
        # This prevents CUDA index out of bounds errors in attention layers
        seq_len = input_ids.shape[1]
        if seq_len >= model_max_length:
            print(f"Warning: Sequence length {seq_len} >= model max {model_max_length}. Truncating to {model_max_length - 1}")
            input_ids = input_ids[:, :model_max_length - 1]
            attention_mask = attention_mask[:, :model_max_length - 1]
            seq_len = input_ids.shape[1]

        # Additional safety: ensure total sequence length (input + output) fits within model
        total_seq_len = seq_len + max_new_tokens
        if total_seq_len >= model_max_length:
            print(f"Warning: Total sequence length {total_seq_len} would exceed model max {model_max_length}. Reducing max_new_tokens.")
            max_new_tokens = max(10, model_max_length - seq_len - 10)  # Extra safety margin, minimum 10 tokens

        # Use torch.no_grad() for inference to save memory and avoid gradient computation issues
        self._model_codegen.eval()  # Ensure model is in eval mode
        if DEBUG_FLAG:
            print(f"DEBUG - Max New Tokens for CodeGen: {max_new_tokens}")

        # Debug: Print tensor information before generation
        if DEBUG_FLAG:
            print(f"DEBUG - input_ids shape: {input_ids.shape}, dtype: {input_ids.dtype}")
            print(f"DEBUG - input_ids min/max: {input_ids.min().item()}/{input_ids.max().item()}")
            print(f"DEBUG - input_ids first 10 values: {input_ids[0, :10].tolist()}")
            print(f"DEBUG - input_ids indices with value >= vocab_size: {(input_ids >= vocab_size).nonzero()}")
            print(f"DEBUG - attention_mask shape: {attention_mask.shape}, dtype: {attention_mask.dtype}")
            print(f"DEBUG - attention_mask sum: {attention_mask.sum().item()}")

        # Check for potential position embedding issues
        if DEBUG_FLAG:
            print(f"DEBUG - seq_len: {seq_len}, max_new_tokens: {max_new_tokens}, total_seq_len: {seq_len + max_new_tokens}")
            print(f"DEBUG - model_max_length: {model_max_length}")

        # Validate that position embeddings won't overflow
        max_position_embeddings = getattr(self._model_codegen.config, 'max_position_embeddings', 2048)
        if seq_len + max_new_tokens > max_position_embeddings:
            print(f"WARNING: Total sequence length {seq_len + max_new_tokens} exceeds position embedding limit {max_position_embeddings}")

        with torch.no_grad():
            if True: #try:
                # Use greedy decoding for stability (do_sample=False avoids numerical issues)
                print("self._model_codegen.generate: Start")
                # Create explicit position_ids to prevent index out of bounds
                #position_ids = torch.arange(input_ids.shape[1], dtype=torch.long, device=self.device).unsqueeze(0)

                # Debug: Check position_ids bounds
                #print(f"DEBUG - position_ids shape: {position_ids.shape}, dtype: {position_ids.dtype}")
                #print(f"DEBUG - position_ids range: {position_ids.min().item()} to {position_ids.max().item()}")
                #print(f"DEBUG - max_position_embeddings: {max_position_embeddings}")

                # Check if position_ids would cause index out of bounds
                if False: #position_ids.max().item() >= max_position_embeddings:
                    if DEBUG_FLAG:
                        print(f"ERROR: position_ids max ({position_ids.max().item()}) >= max_position_embeddings ({max_position_embeddings})")
                        # Truncate position_ids to valid range
                        position_ids = torch.clamp(position_ids, 0, max_position_embeddings - 1)
                        print(f"DEBUG - position_ids clamped to: {position_ids.min().item()} to {position_ids.max().item()}")

                # Check model's actual position embedding size
                actual_pos_emb_size = self._model_codegen.transformer.wpe.num_embeddings if hasattr(self._model_codegen.transformer, 'wpe') else 'unknown'
                if DEBUG_FLAG:
                    print(f"DEBUG - model's actual position embedding size: {actual_pos_emb_size}")

                # Warn if there's a mismatch
                if actual_pos_emb_size != 'unknown' and actual_pos_emb_size < max_position_embeddings:
                    print(f"WARNING: Model's actual position embedding size ({actual_pos_emb_size}) < config max_position_embeddings ({max_position_embeddings})")

                # Check for rotary position embeddings (used in codegen)
                if hasattr(self._model_codegen.transformer, 'wpe'):
                    wpe = self._model_codegen.transformer.wpe
                    if DEBUG_FLAG:
                        print(f"DEBUG - wpe type: {type(wpe)}, shape: {wpe.weight.shape if hasattr(wpe, 'weight') else 'N/A'}")
                        if hasattr(wpe, 'weight'):
                            print(f"DEBUG - wpe weight min/max: {wpe.weight.min().item():.4f} / {wpe.weight.max().item():.4f}")
                else:
                    if DEBUG_FLAG:
                        print("DEBUG - No wpe found, checking for rotary embeddings...")

                import transformers
                if DEBUG_FLAG:
                    print(f"Transformers Version: {transformers.__version__}")
                    print(f"DEBUG - attention_mask.dtype: {attention_mask.dtype}")
                    print(f"DEBUG - attention_mask.device: {attention_mask.device}")
                    #print(f"DEBUG - past_key_values: {type(past_key_values)}")
                    print("DEBUG - n_positions:", self._model_codegen.config.n_positions)
                    print("DEBUG - input length:", input_ids.shape[1])
                    print("DEBUG - max_new_tokens:", max_new_tokens)
                    print("DEBUG - total:", input_ids.shape[1] + max_new_tokens)
                    print(f"DEBUG - {input_ids.shape[1]} + {max_new_tokens} > {self._model_codegen.config.n_positions}")
                    print("CUDA devices:", torch.cuda.device_count())
                    print("Model parameters on device:", next(self._model_codegen.parameters()).device)
                output_ids = self._model_codegen.generate(
                    input_ids,
                    attention_mask=attention_mask,
                    #position_ids=position_ids,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,  # Use greedy decoding for stability
                    num_return_sequences=1,
                    use_cache=False,
                    pad_token_id=self._tokenizer_codegen.eos_token_id
                )
                print("self._model_codegen.generate: Done")
            # except RuntimeError as e:
            #     error_msg = str(e)
            #     is_cuda_error = any(err in error_msg for err in [
            #         "CUBLAS_STATUS_EXECUTION_FAILED",
            #         "index out of bounds",
            #         "device-side assert",
            #         "CUDA error"
            #     ])

                # if is_cuda_error:
                #     # Option 2: Clear GPU cache and retry with even smaller tokens
                #     print(f"CUDA error encountered, clearing cache and retrying...")
                #     try:
                #         if torch.cuda.is_available():
                #             torch.cuda.empty_cache()
                #             gc.collect()
                #     except RuntimeError:
                #         # GPU is in bad state, try to reset
                #         print("GPU in bad state, attempting reset...")
                #         try:
                #             torch.cuda.synchronize()  # Force sync to clear any pending errors
                #         except RuntimeError:
                #             pass  # Ignore if synchronize also fails
                #         gc.collect()

                #     # Retry strategies
                #     retry_strategies = [
                #         {"name": "Strategy 1: Reduced tokens, no sampling",
                #           "max_new_tokens": 64, "do_sample": False},
                #         {"name":"Strategy 2: Greedy with very small tokens",
                #           "max_new_tokens": 32, "do_sample": False},
                #         {"name" :" Strategy 3: Truncated input with re-tokenization",
                #           "max_new_tokens": 64, "do_sample": False, "retokenize": True},
                #     ]

                #     output_ids = None
                #     for i, strategy in enumerate(retry_strategies):
                #         try:
                #             gen_input_ids = input_ids
                #             gen_attention = attention_mask

                #             if strategy.get("retokenize", False):
                #                 # Re-tokenize to get fresh tensors
                #                 print("Re-tokenizing input for retry...")
                #                 fresh_inputs = self._tokenizer_codegen(original_prompt, return_tensors="pt").to(self.device)
                #                 gen_input_ids = fresh_inputs["input_ids"]
                #                 gen_attention = fresh_inputs["attention_mask"]
                #                 # Truncate if needed
                #                 truncate_len = min(256, model_max_length - strategy["max_new_tokens"])
                #                 if gen_input_ids.shape[1] > truncate_len:
                #                     gen_input_ids = gen_input_ids[:, -truncate_len:]
                #                     gen_attention = gen_attention[:, -truncate_len:]
                #             elif strategy.get("truncate", False):
                #                 # Use model's max length or 512, whichever is smaller
                #                 truncate_len = min(512, model_max_length - strategy["max_new_tokens"])
                #                 gen_input_ids = input_ids[:, -truncate_len:] if input_ids.shape[1] > truncate_len else input_ids
                #                 gen_attention = attention_mask[:, -truncate_len:] if attention_mask.shape[1] > truncate_len else attention_mask

                #             output_ids = self._model_codegen.generate(
                #                 gen_input_ids,
                #                 attention_mask=gen_attention,
                #                 max_new_tokens=strategy["max_new_tokens"],
                #                 do_sample=strategy["do_sample"],
                #                 num_return_sequences=1,
                #                 pad_token_id=self._tokenizer_codegen.eos_token_id
                #             )
                #             print(f"Strategy {i+1} {strategy.get('name',' ')} succeeded")
                #             break
                #         except RuntimeError as e2:
                #             print(f"Strategy {i+1} {strategy.get('name',' ')} failed: {e2}")
                #             continue

                #     if output_ids is None:
                #         # Final fallback: Use CPU tensors with the model still on GPU
                #         # (model dispatched with accelerate can't be moved)
                #         print("All GPU strategies failed, trying CPU tensors on GPU model...")
                #         try:
                #             # Put tensors on CPU and let the model handle it
                #             output_ids = self._model_codegen.generate(
                #                 input_ids,  # Keep on GPU, let accelerate handle it
                #                 attention_mask=attention_mask,
                #                 max_new_tokens=32,
                #                 do_sample=False,
                #                 num_return_sequences=1,
                #                 pad_token_id=self._tokenizer_codegen.eos_token_id
                #             )
                #         except Exception as cpu_error:
                #             print(f"CPU tensor fallback also failed: {cpu_error}")
                #             return "# Error: Generation failed"
                # else:
                #     raise e

        if output_ids is None or len(output_ids) == 0 or output_ids[0] is None:
            print("Warning: No output generated, returning empty string")
            return ""

        try:
            generated_text = self._tokenizer_codegen.decode(output_ids[0], skip_special_tokens=True)
        except Exception as decode_error:
            print(f"Warning: Failed to decode output: {decode_error}")
            return ""

        code_prefix = f"{target_lang} code:"
        if code_prefix in generated_text:
            generated_code = generated_text.split(code_prefix, 1)[1].strip()
        else:
            generated_code = generated_text.strip()

        # Post-processing to remove any remaining prompt or unwanted text
        if input_text in generated_code:
            generated_code = generated_code.replace(input_text, "").strip()

        return generated_code

    def compute_baseline(self, num_records: Optional[int] = 100, num_tries: int = 3) -> list:
        """
        Computes a baseline score for a subset of the dataset in two phases.
        Phase 1 (NL->PL) : Finds the best documetnation for the code, from a run of three times. Where the documentation
                generated from code, the generated code is then compared against the ground truth code using the
                AST similarity and the GraphCodeBERTScore (both averaged). The documentation that generates the highest
                score is saved.
       Phase 2 (PL1 -> PL2) : The second phase threre are three steps.
            Step 1 For the C++ code the saved best documentation from phase 1 is taken and given to the model
                   to generate Python Code.
            Step 2 is where the genereated Python code is given to the model to generate C++ code.
            Step 3 is where the generated python code is AST and GraphBertScore compared (both values averaged)
                   with the ground truth C++ code.

            For each record, the phase 2 is done three times, and for each record the best score is saved.

      For the overall python programs in the dataset, all the scores are then averaged.


        Args:
            num_records (Optional[int]): The number of records to process from the dataset.
                                        If None, processes all records.
            num_tries (int): The number of times to attempt code generation and scoring
                             for each record to find the best documentation/code pair.

        Returns:
            list: A list of dictionaries, each containing the hexsha, best AST score (NL->PL),
                  best GraphCodeBERT score (NL->PL), average score (NL->PL), and for Python records,
                  the best AST, GCB, and average scores for the NL->C++->Python translation.
        """
        print(f"\nStarting baseline computation for {num_records if num_records is not None else 'all'} records (each with {num_tries} tries per phase)...")
        # Reset GPU state before starting
        if torch.cuda.is_available():
            print(f"\nSynchronizing CUDA and empty cache")
            torch.cuda.synchronize()
            torch.cuda.empty_cache()

        results = []
        processed_count = 0

        # Use the cached dataset instead of streaming
        print(f"Using cached dataset with {len(self._cached_dataset)} samples...")

        for i in range(len(self._cached_dataset)):
            if num_records is not None and processed_count >= num_records:
                break

            record = self._cached_dataset[i]

            if not record.get('is_processable_code', True):
                continue # Skip unprocessable records

            processed_count += 1
            print(f"Processing record {processed_count}/{num_records if num_records else 'all'} (language: {record.get('language', 'unknown')})...")

            original_hexsha = record['hexsha']
            original_code = record['content']
            language = record['language']

            # --- Phase 1: NL -> PL1 (Documentation to Code) --- (NL->PL1 for the record's actual language)
            best_ast_score_nl_pl = -1.0
            best_graphcodebert_score_nl_pl = -1.0
            best_average_score_nl_pl = -1.0
            best_documentation = "" # This stores documentation for the best NL->PL1 result

            print(f"  Generating documentation for {original_hexsha[:8]}...")
            # Generate documentation for Phase 1
            generated_doc_phase1 = self._documentation_generator.generate_documentation(original_code, max_length=256)

            print(f"  Generating code from documentation for {language}...")
            # Generate code from the generated documentation using the base model
            generated_code_phase1 = self._generate_code_from_model(generated_doc_phase1, language)

            if not generated_code_phase1.strip():
                print(f"  Warning: Empty generated code, skipping...")
                continue

            print(f"  Comparing ASTs...")
            try:
                original_ast = self._ast_processor.generate_ast(original_code, language)
                generated_ast_phase1 = self._ast_processor.generate_ast(generated_code_phase1, language)
                ast_similarity_phase1 = self._ast_processor.compare_ast(original_ast, generated_ast_phase1)
            except Exception as e:
                print(f"  AST comparison error: {e}")
                ast_similarity_phase1 = 0.0

            print(f"  Computing semantic similarity GraphCodeBERTScore...")
            try:
                semantic_similarity_phase1 = self._graphcodebert_scorer.score(original_code, generated_code_phase1)
            except RuntimeError as e:
                print(f"  GraphCodeBERTScore comparison CUDA error: {e}")
                semantic_similarity_phase1 = 0.0
            except Exception as e:
                print(f"  GraphCodeBERTScore comparison error: {e}")
                semantic_similarity_phase1 = 0.0

            current_average_score_phase1 = (ast_similarity_phase1 + semantic_similarity_phase1) / 2.0
            print(f"\t AST Similarity Score: {ast_similarity_phase1:0.4f}, GraphCodeBERTScore: {semantic_similarity_phase1:0.4f}, Average: {current_average_score_phase1:0.4f}")

            if current_average_score_phase1 > best_average_score_nl_pl:
                best_average_score_nl_pl = current_average_score_phase1
                best_ast_score_nl_pl = ast_similarity_phase1
                best_graphcodebert_score_nl_pl = semantic_similarity_phase1
                best_documentation = generated_doc_phase1 # Store the doc that led to the best score

            # Initialize Phase 2 scores to None
            best_ast_score_nl_pl1_pl2 = None
            best_gcb_score_nl_pl1_pl2 = None
            best_avg_score_nl_pl1_pl2 = None

            # --- Phase 2: NL -> PL1 -> PL2 (Documentation -> Python -> C++) for C++ records ---
            if language.lower() in ['c++', 'cpp']:
                current_best_avg_phase2 = -1.0
                temp_best_ast_phase2 = 0.0
                temp_best_gcb_phase2 = 0.0

                if not best_documentation.strip(): # Skip phase 2 if no good doc was found in phase 1
                    # print(f"Skipping Phase 2 for {original_hexsha} due to no valid documentation from Phase 1.")
                    pass
                else:
                    best_llm_judge_score_for_intermediate_python = -1.0
                    best_generated_python_code_for_phase2 = ""

                    # Step 1: Generate C++ code from best_documentation (from Phase 1) and use LLM Judge to pick the best
                    for _ in range(num_tries):
                        import re
                        documentation_for_pl2 = re.sub(r'cpp|c\+\+', 'Python', best_documentation, flags=re.IGNORECASE)
                        generated_python_candidate = self._generate_code_from_model(documentation_for_pl2, 'python',is_py_to_cpp=False)
                        if generated_python_candidate.strip():
                            llm_judge_current_score = self._llm_judge.qwen_code_judge(documentation_for_pl2,
                            generated_python_candidate)
                            if llm_judge_current_score == 0.0:
                                print(f"Jude failed: \n\n\t***Documentation:{documentation_for_pl2}, \n\n\t****Generated Code:{generated_python_candidate}")

                            if llm_judge_current_score > best_llm_judge_score_for_intermediate_python:
                                best_llm_judge_score_for_intermediate_python = llm_judge_current_score
                                best_generated_python_code_for_phase2 = generated_python_candidate

                    generated_python_code_from_doc = best_generated_python_code_for_phase2

                    if not generated_python_code_from_doc.strip():
                        continue # Skip if no good C++ code was generated even after tries

                    #for _ in range(num_tries): This we do not have to do thrice
                    if True:
                        # Step 2: Generate C++ code from the best generated Python code
                        final_generated_cpp_code = self._generate_code_from_model(generated_python_code_from_doc, 'cpp', is_py_to_cpp=True)

                        if not final_generated_cpp_code.strip():
                            continue

                        # Step 3: Compare final generated Python code with original Python code
                        try:
                            original_ast = self._ast_processor.generate_ast(original_code, language)
                            generated_ast_phase2 = self._ast_processor.generate_ast(final_generated_cpp_code, language)
                            ast_similarity_phase2 = self._ast_processor.compare_ast(original_ast, generated_ast_phase2)
                        except Exception as e:
                            print(f"Forcing AST Similarity to : AST comparison for {original_hexsha} (Phase 2) due to error: {e}")
                            ast_similarity_phase2 = 0.0

                        try:
                            semantic_similarity_phase2 = self._graphcodebert_scorer.score(original_code, final_generated_cpp_code)
                        except RuntimeError as e:
                            print(f"Forcing  GCB comparison for {original_hexsha} (Phase 2) to 0 due to CUDA error: {e}")
                            semantic_similarity_phase2 = 0.0
                        except Exception as e:
                            print(f"Forcing GCB comparison for {original_hexsha} (Phase 2) to 0 due to error: {e}")
                            semantic_similarity_phase2 = 0.0

                        current_average_score_phase2 = (ast_similarity_phase2 + semantic_similarity_phase2) / 2.0

                        if current_average_score_phase2 > current_best_avg_phase2:
                            current_best_avg_phase2 = current_average_score_phase2
                            temp_best_ast_phase2 = ast_similarity_phase2
                            temp_best_gcb_phase2 = semantic_similarity_phase2

                    if current_best_avg_phase2 > -1.0: # If at least one successful generation occurred in Phase 2
                        best_ast_score_nl_pl1_pl2 = temp_best_ast_phase2
                        best_gcb_score_nl_pl1_pl2 = temp_best_gcb_phase2
                        best_avg_score_nl_pl1_pl2 = current_best_avg_phase2

            # Update the FilteredDataset's internal caches with the best results from both phases
            if best_average_score_nl_pl > -1.0: # Only update if Phase 1 was successful
                self._filtered_dataset.update_documentation(original_hexsha, best_documentation)
                self._filtered_dataset.update_scores(
                    record_identifier=original_hexsha,
                    ast_score=best_ast_score_nl_pl,
                    graphcodebert_score=best_graphcodebert_score_nl_pl,
                    average_score=best_average_score_nl_pl,
                    python_translation_ast_score=best_ast_score_nl_pl1_pl2,
                    python_translation_gcb_score=best_gcb_score_nl_pl1_pl2,
                    python_translation_average_score=best_avg_score_nl_pl1_pl2
                )

                # Option 2: Clear GPU cache after each record to prevent memory fragmentation
                if torch.cuda.is_available():
                    try:
                        print("\tClear GPU cache after each record to prevent memory fragmentation")
                        torch.cuda.empty_cache()
                        gc.collect()
                    except RuntimeError:
                        print("GPU is in bad state, skip cache clearing")
                        pass

                print(f"\tRecording result: 'hexsha': {original_hexsha[:10]}, 'nl_pl_best_ast_score': {best_ast_score_nl_pl:0.4f}, 'nl_pl_best_graphcodebert_score': {best_graphcodebert_score_nl_pl:0.4f}, 'nl_pl_best_average_score': {best_average_score_nl_pl:0.4f}")

                record_result = {
                    'hexsha': original_hexsha,
                    'nl_pl_best_ast_score': best_ast_score_nl_pl,
                    'nl_pl_best_graphcodebert_score': best_graphcodebert_score_nl_pl,
                    'nl_pl_best_average_score': best_average_score_nl_pl
                }

                if language.lower() in ['c++', 'cpp']:
                    record_result.update({
                        'nl_pl1_pl2_best_ast_score': best_ast_score_nl_pl1_pl2,
                        'nl_pl1_pl2_best_graphcodebert_score': best_gcb_score_nl_pl1_pl2,
                        'nl_pl1_pl2_best_average_score': best_avg_score_nl_pl1_pl2
                    })
                    print(f"\tRecording NL->PL1->PL2 result: 'hexsha': {original_hexsha[:10]}, 'nl_pl1_pl2_best_ast_score': {best_ast_score_nl_pl1_pl2:0.4f}, 'nl_pl1_pl2_best_average_score': {best_avg_score_nl_pl1_pl2:0.4f}")
                results.append(record_result)
                processed_count += 1
                print(f"\nProcessed {processed_count}/{num_records if num_records is not None else 'all'} records. Current record {original_hexsha[:10]}... " +
                      f"NL->PL Avg Score: {best_average_score_nl_pl:.4f}" +
                      (f", NL->C++->Py Avg Score: {best_avg_score_nl_pl1_pl2:.4f}" if best_avg_score_nl_pl1_pl2 is not None else ""))

        return results


In [ ]:
from typing import Optional # Added import
import os

# Set environment variables to help with CUDA errors
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # For better error reporting

# Check if GPU is in a usable state
def is_gpu_usable():
    """Check if GPU is in a usable state."""
    if not torch.cuda.is_available():
        return False
    try:
        # Try a simple CUDA operation
        test_tensor = torch.zeros(1, device='cuda')
        _ = test_tensor + 1
        return True
    except RuntimeError:
        return False

print("\n--- Starting Baseline Evaluation for the entire dataset ---")

# --- BEGIN FIX: Ensure all singletons are fully re-initialized ---
# This is crucial in interactive environments where class definitions might be re-run
# or partial executions can leave singletons in an inconsistent state.
# Reset Singleton states for all classes involved to ensure clean re-initialization.
print("Resetting singleton states for BaselineData and its dependencies...")
for cls_to_reset in [BaselineData, CodeDocumentationGenerator, AST, GraphCodeBERTScorer, FilteredDataset, LLMJudge, QwenModelBase]:
    if cls_to_reset in SingletonMeta._instances:
        del SingletonMeta._instances[cls_to_reset]
    if hasattr(cls_to_reset, '_initialized'):
        cls_to_reset._initialized = False
    # QwenModelBase has a specific initialization flag
    if hasattr(cls_to_reset, '_initialized_qwen'):
        cls_to_reset._initialized_qwen = False
print("Singleton states reset.")
# --- END FIX ---

# Check GPU health and force CPU if needed
if torch.cuda.is_available() and not is_gpu_usable():
    print("WARNING: GPU is in bad state, forcing CPU mode for this run")
    BaselineData._preferred_device = 'cpu'

# Instantiate the BaselineData singleton
#TODO: Change this value to control how many records in the dataset. The dataset is cached before processing it thorugh the models. Each cached dataset has this count in it same so every time this changes a new cache is created. This can keep eating space if this count changes frequenty.
num_samples_for_evaluation = 100
baseline_evaluator = BaselineData(num_samples = num_samples_for_evaluation)

# Set num_records_for_full_eval to None to process all records, or specify a number.
num_records_for_full_eval: Optional[int] = None # Process all records
# If you want to process a specific number of records for testing, uncomment and set the value:
num_records_for_full_eval = 10

num_generation_tries_eval = 3 # Number of tries for code generation per record

full_baseline_results = baseline_evaluator.compute_baseline(
    num_records=num_records_for_full_eval, # Pass None here
    num_tries=num_generation_tries_eval
)

# Calculate and display accuracy metrics
overall_scores_nl_pl = []
python_scores_nl_pl = []
cpp_scores_nl_pl = []

overall_scores_nl_pl1_pl2 = [] # New list for Phase 2 scores
python_scores_nl_pl1_pl2 = [] # New list for Phase 2 scores (only for Python)

# Assuming `FilteredDataset` holds the up-to-date information after `compute_baseline`.
# We'll use the filtered_dataset_instance managed by BaselineData to retrieve language info.
# Make sure to reset its iterator to start fresh, and then iterate through it
# to get the language for each hexsha that was processed and has scores in `full_baseline_results`.
filtered_dataset_instance_for_metrics = baseline_evaluator._filtered_dataset
filtered_dataset_instance_for_metrics.reset_iterator()

# Create a dictionary to quickly look up scores by hexsha
results_by_hexsha = {res['hexsha']: res for res in full_baseline_results}

# Iterate through the filtered_dataset and collect scores for processed records
processed_for_metrics_count = 0
for record in filtered_dataset_instance_for_metrics:
    # We only care about records that were actually processed and have results
    hexsha = record.get('hexsha')
    if hexsha in results_by_hexsha:
      score_data = results_by_hexsha[hexsha]

      # Collect scores for Phase 1 (NL->PL)
      avg_score_nl_pl = score_data['nl_pl_best_average_score']
      overall_scores_nl_pl.append(avg_score_nl_pl)

      language = record.get('language')
      if language.lower() in ['c++', 'cpp']:
        python_scores_nl_pl.append(avg_score_nl_pl)
        # Collect scores for Phase 2 (NL->PL1->PL2) for Python records
        if score_data['nl_pl1_pl2_best_average_score'] is not None:
          avg_score_nl_pl1_pl2 = score_data['nl_pl1_pl2_best_average_score']
          overall_scores_nl_pl1_pl2.append(avg_score_nl_pl1_pl2)
          python_scores_nl_pl1_pl2.append(avg_score_nl_pl1_pl2)
      elif language.lower() in ['python']:
        cpp_scores_nl_pl.append(avg_score_nl_pl)

      processed_for_metrics_count += 1
      # If a specific number of records was processed, we stop once we have enough scores for metrics
      if num_records_for_full_eval is not None and processed_for_metrics_count >= num_records_for_full_eval:
        break

# Calculate averages for Phase 1 (NL->PL)
overall_average_accuracy_nl_pl = sum(overall_scores_nl_pl) / len(overall_scores_nl_pl) if overall_scores_nl_pl else 0.0
python_average_accuracy_nl_pl = sum(python_scores_nl_pl) / len(python_scores_nl_pl) if python_scores_nl_pl else 0.0
cpp_average_accuracy_nl_pl = sum(cpp_scores_nl_pl) / len(cpp_scores_nl_pl) if cpp_scores_nl_pl else 0.0

# Calculate averages for Phase 2 (NL->PL1->PL2)
overall_average_accuracy_nl_pl1_pl2 = sum(overall_scores_nl_pl1_pl2) / len(overall_scores_nl_pl1_pl2) if overall_scores_nl_pl1_pl2 else 0.0
python_average_accuracy_nl_pl1_pl2 = sum(python_scores_nl_pl1_pl2) / len(python_scores_nl_pl1_pl2) if python_scores_nl_pl1_pl2 else 0.0

print("\n--- Final Accuracy Baselines ---")
print(f"\nPhase 1 (NL->PL) Evaluation:")
print(f"Overall Average Code Generation Accuracy (NL->PL): {overall_average_accuracy_nl_pl:.4f} (based on {len(overall_scores_nl_pl)} records)")
print(f"Python Average Code Generation Accuracy (NL->PL):    {python_average_accuracy_nl_pl:.4f} (based on {len(python_scores_nl_pl)} records)")
print(f"C++ Average Code Generation Accuracy (NL->PL):       {cpp_average_accuracy_nl_pl:.4f} (based on {len(cpp_scores_nl_pl)} records)")

print(f"\nPhase 2 (NL->Python->C++) Evaluation for Python:")
print(f"Redundant: Overall Average Code Generation Accuracy (NL->Python->C++): {overall_average_accuracy_nl_pl1_pl2:.4f} (based on {len(overall_scores_nl_pl1_pl2)} records)")
print(f"Python Average Code Generation Accuracy (NL->Python->C++):    {python_average_accuracy_nl_pl1_pl2:.4f} (based on {len(python_scores_nl_pl1_pl2)} records)")



--- Starting Baseline Evaluation for the entire dataset ---
Resetting singleton states for BaselineData and its dependencies...
Singleton states reset.
Initializing BaselineData singleton using device: cpu...
QwenModelBase using device: cpu for model Qwen/Qwen2.5-Coder-7B-Instruct
CodeDocumentationGenerator using device: cpu
Initializing GraphCodeBERTScorer...
Using device: cpu


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


GraphCodeBERTScorer initialized successfully.
Initializing FilteredDataset singleton...
Loading Python dataset...


Resolving data files:   0%|          | 0/144 [00:00<?, ?it/s]

...

Code generation model loaded.
BaselineData initialized successfully.

Starting baseline computation for 50 records (each with 3 tries per phase)...
Resetting dataset streams and iterators...
Loading Python dataset...


Resolving data files:   0%|          | 0/144 [00:00<?, ?it/s]

Loading C++ dataset...


Resolving data files:   0%|          | 0/110 [00:00<?, ?it/s]

Starting iteration over combined dataset (alternating Python and C++) via __iter__...
Documentation for record 'd99a1e98eccb58cbc0c0cef6e9e6702f33461b0e' updated locally.
Scores for record 'd99a1e98eccb58cbc0c0cef6e9e6702f33461b0e' updated locally: NL->PL Avg=0.7203
Processed 1/50 records. Current record d99a1e98ec... NL->PL Avg Score: 0.7203
Documentation for record '4f1d9e36be378fb394dc7da2e912f46bf7c71a18' updated locally.
Scores for record '4f1d9e36be378fb394dc7da2e912f46bf7c71a18' updated locally: NL->PL Avg=0.5712
Processed 2/50 records. Current record 4f1d9e36be... NL->PL Avg Score: 0.5712
Documentation for record 'd99a20277c32bb1e28312f42ab6d732f38323169' updated locally.
Scores for record 'd99a20277c32bb1e28312f42ab6d732f38323169' updated locally: NL->PL Avg=0.5216
Processed 3/50 records. Current record d99a20277c... NL->PL Avg Score: 0.5216


# Create LORA of codegen-multi-350B for code generation where the input is code documentation and output is the code.

### Load the Codegen-350M-multi model
#### Lora Adapt
#### For the C++ and python code, get one, get the documentation, give model the documentation and let it generate the code.
#### The generated code must be same as the code for which the documetation was generated.

#### The second set of train is where we give documentation of a python code and have the model generate C++ Code:
 1. Take C++ code.
 2. generate documentation,
 3. Generated documentation input to model to genreate Python Code
 4. Python Code as input to model to generate C++ Code
 5. C++ code comparison gives the loss.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name_codegen = "Salesforce/codegen-350M-multi"

# Load tokenizer and model for codegen
tokenizer_codegen = AutoTokenizer.from_pretrained(model_name_codegen)
# Ensure pad_token_id is explicitly set for the global tokenizer
if tokenizer_codegen.pad_token_id is None:
    tokenizer_codegen.pad_token_id = tokenizer_codegen.eos_token_id

model_codegen = AutoModelForCausalLM.from_pretrained(
    model_name_codegen,
    torch_dtype=torch.float32,
    device_map="auto",
    trust_remote_code=False
)
tokenizer_codegen.pad_token = tokenizer_codegen.eos_token
model_codegen.config.pad_token_id = model_codegen.config.eos_token_id

print("Printing modules in CodeGen-350m-multi-model")
for name, module in model_codegen.named_modules():
    print(f"\t{name}")

### LORA Adaptation for `codegen-350M-multi`

Set up Low-Rank Adaptation (LORA) for the `codegen-350M-multi` model. This allows us to fine-tune the model efficiently without modifying all its parameters. We'll specify the LORA configuration, including the rank (`r`), alpha (`lora_alpha`), dropout (`lora_dropout`), and target modules (the layers to apply LORA to).


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType


# Define LORA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, # For Causal Language Modeling
    inference_mode=False,
    r=8, # Rank of the update matrices, Should this be 16? TODO
    lora_alpha=32, # Scaling factor, should this be 32: TODO
    lora_dropout=0.1, # Dropout probability, should this be small 0.05: TODO
    target_modules=["qkv_proj", "out_proj"] # Apply LORA to query, key, value and out projections
)

# Apply LORA to the codegen model
model_codegen_lora = get_peft_model(model_codegen, lora_config)

print("\n"+"*"*30+"\nLORA adapted model summary:")
model_codegen_lora.print_trainable_parameters()
print("\n"+"*"*30)

In [ ]:
import torch

print(torch.cuda.memory_summary())

In [ ]:
!nvidia-smi

### Train the LORA Model (NL->PL1): Phase 1

Train LORA using the filtered data set for C++ Code. Where for each record the model is trianed thrice to generate C++ code from its documentation.
The ground truth C++ code is in the dataset and then there is the generated C++ Code for comparison.

Comparison is performed using the AST coparision using the GraphCodeBertScore (for AST similarity) and CodeBERTScore for code similarity.

After this training the model is expected to be able to convert a documenation to C++ code.

Set up the `Trainer` from the `transformers` library to fine-tune our LORA-adapted `codegen` model.
Define `TrainingArguments` to control the training process, such as the number of epochs, learning rate, and logging strategy.
Use a `DataCollatorForLanguageModeling` to handle batching and masking for language modeling tasks.

To save the best model based on validation accuracy , you would typically include a validation set and a custom `TrainerCallback` or configure `save_strategy='epoch'` and `load_best_model_at_end=True` with a specified `metric_for_best_model` in `TrainingArguments`.


In [ ]:
import torch
from datasets import Dataset
from tqdm import tqdm

# --- BEGIN FIX: Ensure all singletons are fully re-initialized before use in this cell ---
# This is crucial in interactive environments where class definitions might be re-run
# or partial executions can leave singletons in an inconsistent state.
# Reset Singleton states for all classes involved to ensure clean re-initialization.
print("Resetting singleton states for BaselineData and its dependencies")
for cls_to_reset in [BaselineData, CodeDocumentationGenerator, AST, GraphCodeBERTScorer, FilteredDataset, LLMJudge, QwenModelBase]:
    if cls_to_reset in SingletonMeta._instances:
        del SingletonMeta._instances[cls_to_reset]
    if hasattr(cls_to_reset, '_initialized'):
        cls_to_reset._initialized = False
    # QwenModelBase has a specific initialization flag
    if hasattr(cls_to_reset, '_initialized_qwen'):
        cls_to_reset._initialized_qwen = False
print("Singleton states reset in i0YOCFPWoYVV.")
# --- END FIX ---

# Instantiate the BaselineData singleton. This call will now guarantee a fresh instance.
baseline_evaluator = BaselineData(num_samples = num_samples_for_evaluation)
print("BaselineData instance ensured for LORA training dataset generation.")

# Assuming baseline_evaluator is already initialized
filtered_dataset_instance = baseline_evaluator._filtered_dataset
documentation_generator = baseline_evaluator._documentation_generator

# Collect C++ examples for LORA training (NL->C++)
lora_nl_cpp_examples = []
MAX_RECORDS_FOR_LORA_NL_CPP = 10 # Define a limit for the LORA dataset, adjust as needed

print(f"Collecting {MAX_RECORDS_FOR_LORA_NL_CPP} C++ records for LORA NL->C++ training...")

# Get a fresh iterator for the C++ stream only
cpp_stream_iterator = filtered_dataset_instance.get_cpp_stream_iterator()

processed_cpp_records = 0
for record in tqdm(cpp_stream_iterator, desc="Processing C++ records for LORA NL->C++ data"):
    if processed_cpp_records >= MAX_RECORDS_FOR_LORA_NL_CPP:
        print(f"Reached max records for LORA NL->C++ training: {MAX_RECORDS_FOR_LORA_NL_CPP}")
        break

    if not record['is_processable_code'] or record['language'].lower() in ['c++', 'cpp']:
        continue

    # Use existing documentation from the record (generated by BaselineData if available)
    # If not, generate new documentation as a fallback
    documentation = record.get('documentation')
    if not documentation or not documentation.strip():
        # Fallback to generating documentation if not present or empty
        documentation = documentation_generator.generate_documentation(record['content'])
        if not documentation.strip():
            print(f"Skipping record {record.get('hexsha', 'unknown')} due to inability to generate/find C++ documentation.")
            continue

    lora_nl_cpp_examples.append({
        "document": documentation,
        "code": record['content'] # The ground truth C++ code
    })
    processed_cpp_records += 1
    print(f"Processed {processed_cpp_records} C++ records for LORA NL->C++")

if lora_nl_cpp_examples:
    training_dataset = Dataset.from_list(lora_nl_cpp_examples)
    print(f"Successfully created training_dataset with {len(training_dataset)} examples for NL->C++.")
else:
    training_dataset = Dataset.from_list([])
    print("No suitable training examples were generated for NL->C++.")

from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Tokenize the training dataset
def tokenize_function(examples):
    # Combine document (prompt) and code (target) into a single text string for Causal LM training
    # The tokenizer will then process this combined text.
    # DataCollatorForLanguageModeling with mlm=False will automatically handle shifting labels
    # so that the loss is only computed on the target code part.
    full_texts = []
    for doc, code in zip(examples['document'], examples['code']):
        prompt = f"Generate C++ code based on the following documentation:\n{doc}\nC++ code:\n"
        full_texts.append(prompt + code)
    return tokenizer_codegen(full_texts, truncation=True, max_length=512)

# Map the training_dataset with the updated tokenize_function
tokenized_training_dataset = training_dataset.map(tokenize_function, batched=True)

# Data collator for language modeling (will handle padding and labels for CausalLM)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer_codegen, mlm=False)

print("Created Training arguments for LORA")
# Define training arguments
training_args = TrainingArguments(
    output_dir="./codegen_lora_results",
    per_device_train_batch_size=2, # Adjust based on GPU memory
    gradient_accumulation_steps=4, # Increase if batch size is small
    num_train_epochs=3, # Number of training epochs
    learning_rate=2e-4,
    logging_dir="./codegen_lora_logs",
    logging_steps=10,
    save_strategy="epoch", # Save checkpoint every epoch
    save_total_limit=1, # Only keep the best model
    # evaluation_strategy="epoch", # Uncomment if you have a validation set
    # load_best_model_at_end=True, # Uncomment if you have a validation set
    # metric_for_best_model="eval_loss", # Uncomment if you have a validation set
)

print("Created Trainer for LORA")
# Initialize Trainer
trainer = Trainer(
    model=model_codegen_lora,
    args=training_args,
    train_dataset=tokenized_training_dataset,
    # eval_dataset=tokenized_validation_dataset, # Uncomment if you have a validation set
    data_collator=data_collator,
)

# Start training
print("Starting LORA training...")
trainer.train()
print("LORA training complete.")

# Save the final LORA model (or the best model if validation is used)
model_codegen_lora.save_pretrained("codegen_lora_adapter")
print("LORA adapter model saved to 'codegen_lora_adapter'.")

Initializing BaselineData for LORA training dataset generation...


AttributeError: 'BaselineData' object has no attribute '_filtered_dataset'

### LORA Training Phase 2 (PL1->PL2): Python to C++ Code Generation

This phase aims to train the LORA-adapted model to translate C++ code back into Python. The process involves:

1.  **Extract C++ Data**: Obtain records from the C++ stream of the `FilteredDataset`.
2.  **Generate  Documentation**: For each original C++ code snippet, generate its documentation using `CodeDocumentationGenerator`.
3.  **Generate Intermediate Python Code**: Using the *base* `codegen-350M-multi` model (not the LORA-adapted one), generate Python code based on the C++ documentation. This generated Python code will serve as the *input prompt* for the LORA training.
4.  **LORA Training Pair**: The LORA model will be trained on pairs of (`generated_python_code`, `original_cpp_code`), effectively learning to translate from Python back to C++. The goal is for the LORA model to output C++ code that has high similarity (AST and GraphCodeBERT) to the original C++ code.

This is a specific form of code-to-code translation where the Python code serves as the source and C++  code as the target.


#### NL -> PL1 from fine tuned cogden model

In [ ]:
import torch
from datasets import Dataset
from tqdm import tqdm
import os

print("Generating LORA training dataset for Python to C++ translation...")

# --- Reload models if not accessible (User Requirement 3) ---
# Ensure tokenizer_codegen and model_codegen are loaded
if 'model_codegen' not in globals() or 'tokenizer_codegen' not in globals():
    print("Reloading base codegen model and tokenizer...")
    from transformers import AutoTokenizer, AutoModelForCausalLM
    model_name_codegen = "Salesforce/codegen-350M-multi"
    tokenizer_codegen = AutoTokenizer.from_pretrained(model_name_codegen)
    if tokenizer_codegen.pad_token_id is None:
        tokenizer_codegen.pad_token_id = tokenizer_codegen.eos_token_id

    # Determine device globally for this cell
    _current_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Reloaded base codegen model using device: {_current_device}")

    if _current_device == 'cuda':
        model_codegen = AutoModelForCausalLM.from_pretrained(
            model_name_codegen,
            torch_dtype=torch.float16,
            device_map="auto"
        )
    else:
        model_codegen = AutoModelForCausalLM.from_pretrained(model_name_codegen)
        model_codegen.to(_current_device)

# Ensure model_codegen_lora is loaded
if 'model_codegen_lora' not in globals():
    print("Re-applying LORA config to codegen model...")
    from peft import LoraConfig, get_peft_model, PeftModel, TaskType
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=8,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=["qkv_proj", "out_proj"]
    )
    print("LORA Config created")
    # If _current_device is not set from reload, assume CPU for safety or re-detect
    if '_current_device' not in locals():
        _current_device = 'cuda' if torch.cuda.is_available() else 'cpu'

    model_codegen_lora = get_peft_model(model_codegen, lora_config)
    # Attempt to load saved LORA adapter weights if they exist (User Requirement 3)
    if os.path.exists("codegen_lora_adapter"):
        print("Loading LORA adapter weights from 'codegen_lora_adapter'...")
        try:
            # Need to load the base model first, then the peft model
            model_codegen_lora = PeftModel.from_pretrained(model_codegen, "codegen_lora_adapter")
        except Exception as e:
            print(f"Error loading LORA adapter weights: {e}. Proceeding with fresh LORA weights.")
    else:
        print("Warning: 'codegen_lora_adapter' not found. Training will start with fresh LORA weights.")

# Helper function to generate code using the LORA-tuned model (User Requirement 2)
def generate_code_from_lora_model(model, tokenizer, documentation: str, target_lang: str, device: str) -> str:
    prompt = f"Convert the following documentation to {target_lang} code:\n{documentation}\n{target_lang} code:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    output_ids = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=256,
        do_sample=True,
        top_k=50,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id
    )
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    code_prefix = f"{target_lang} code:"
    if code_prefix in generated_text:
        generated_code = generated_text.split(code_prefix, 1)[1].strip()
    else:
        generated_code = generated_text.strip()

    if documentation in generated_code:
        generated_code = generated_code.replace(documentation, "").strip()
    return generated_code


processed_python_records = 0
MAX_RECORDS_FOR_LORA_CPP_TO_PY = 10 # Limit for demonstration, adjust as needed

# Ensure baseline_evaluator is initialized to use its components
# If it hasn't been run yet, this will initialize it.
baseline_evaluator = BaselineData()

# Get a fresh iterator for the C++ stream only
pl2_stream_iterator = baseline_evaluator._filtered_dataset.get_cpp_stream_iterator()

for record in tqdm(pl2_stream_iterator, desc="Processing C++ records for Python to C++ data"):
    if processed_python_records >= MAX_RECORDS_FOR_LORA_CPP_TO_PY:
        print(f"Reached max records for Python to C++ LORA training: {MAX_RECORDS_FOR_LORA_CPP_TO_PY}")
        break

    if not record['is_processable_code'] or record['language'].lower() != 'python':
        continue

    original_pl2_code = record['content']

    # Step 1: Use the python documentation from the dataset itself (User Requirement 1)
    pl2_documentation = record.get('documentation', '')
    if not pl2_documentation.strip():
        print(f"Warning: C++ documentation not found in record {record.get('hexsha', 'unknown')}. Re-generating for this record as a fallback.")
        # Fallback: Generate documentation if not present in the record (against 'not recreate' instruction, but necessary if empty)
        pl2_documentation = baseline_evaluator._documentation_generator.generate_documentation(original_pl2_code, max_length=256)
        if not pl2_documentation.strip(): # If still no documentation, skip
            print(f"Skipping record {record.get('hexsha', 'unknown')} due to inability to generate/find Python documentation.")
            continue

    best_llm_judge_score_for_intermediate_pl1 = -1.0
    best_generated_pl1_code_for_phase2 = ""
    num_tries_llm_judge = 3 # Generate PL2 code three times as requested

    # Step 2: Use the fine-tuned LORA model to generate C++ code and use LLM Judge to pick the best
    for _ in range(num_tries_llm_judge):
        generated_pl1_candidate = generate_code_from_lora_model(model_codegen_lora, tokenizer_codegen, pl2_documentation, target_lang='python', device=baseline_evaluator.device)
        if generated_pl1_candidate.strip():
            # Call the LLM Judge
            llm_judge_current_score = baseline_evaluator._llm_judge.qwen_code_judge(pl2_documentation, generated_pl1_candidate)
            if llm_judge_current_score > best_llm_judge_score_for_intermediate_pl1:
                best_llm_judge_score_for_intermediate_pl1 = llm_judge_current_score
                best_generated_pl1_code_for_phase2 = generated_pl1_candidate

    generated_pl1_code = best_generated_pl1_code_for_phase2

    if generated_pl1_code.strip(): # Only add if C++ code was successfully generated and deemed best
        # The training pair will be (generated_cpp_code, original_python_code)
        pl1_to_pl2_training_examples.append({
            "pl1_code_prompt": generated_pl1_code,
            "pl2_code_target": original_pl2_code
        })
        processed_python_records += 1

# Convert the list of dictionaries to a Hugging Face Dataset
if pl1_to_pl2_training_examples:
    pl1_to_pl2_training_dataset = Dataset.from_list(pl1_to_pl2_training_examples)
    print(f"Successfully created pl1_to_pl2_training_dataset with {len(pl1_to_pl2_training_dataset)} examples.")
    print("First training example for Python to C++:")
    print(pl1_to_pl2_training_dataset[0])
else:
    pl1_to_pl2_training_dataset = Dataset.from_list([])
    print("No suitable training examples were generated for C++ to Python.")

#### PL1 -> PL2 fine tuning

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# Tokenize the new training dataset for C++ to Python
def tokenize_function_cpp_to_py(examples):
    # Combine generated C++ code (prompt) and original Python code (target) into a single text string
    full_texts = []
    for pl1_code, pl2_code in zip(examples['pl1_code_prompt'], examples['pl2_code_target']):
        prompt = f"Convert the following C++ code to Python:\n{pl1_code}\nPython code:\n"
        full_texts.append(prompt + pl2_code)
    return tokenizer_codegen(full_texts, truncation=True, max_length=512)

# Map the new training_dataset with the updated tokenize_function
tokenized_pl1_to_pl2_training_dataset = pl1_to_pl2_training_dataset.map(tokenize_function_cpp_to_py, batched=True)

# Data collator for language modeling (will handle padding and labels for CausalLM)
# Use the same data_collator as before

# Define training arguments (can reuse or modify from previous phase)
# It's good practice to have separate output directories for different training phases
training_args_py_to_cpp = TrainingArguments(
    output_dir="./codegen_lora_results_py_to_cpp",
    per_device_train_batch_size=2, # Adjust based on GPU memory
    gradient_accumulation_steps=4, # Increase if batch size is small
    num_train_epochs=3, # Number of training epochs
    learning_rate=2e-4,
    logging_dir="./codegen_lora_logs_py_to_cpp",
    logging_steps=10,
    save_strategy="epoch", # Save checkpoint every epoch
    save_total_limit=1, # Only keep the best model
)

# Initialize Trainer for the new training phase
trainer_py_to_cpp = Trainer(
    model=model_codegen_lora, # Continue training on the already LORA-adapted model
    args=training_args_py_to_cpp,
    train_dataset=tokenized_pl1_to_pl2_training_dataset,
    data_collator=data_collator,
)

# Start training
print("Starting LORA training for Python to C++ translation...")
trainer_py_to_cpp.train()
print("LORA training for Python to C++ complete.")

# Save the final LORA model (or the best model if validation is used)
model_codegen_lora.save_pretrained("codegen_lora_adapter_cpp_to_py")
print("LORA adapter model saved to 'codegen_lora_adapter_cpp_to_py'.")

NameError: name 'cpp_to_python_training_dataset' is not defined

# Validation of the LORA fine tuned model

## Dataset used:
## Step 1: Generate documenation of the python code, unless it is already availabel in the dataset.
## Step 2: Generate C++ code from the documentation
## Step 3: Use the LORA fine tuned codegen-350 multi to generate python code for the C++ Code.
## Step 4: for the genreated python code and the ground truch python code, use AST compare and the GraphCodeBERTScore to compare the two. Avreage the two scores for the record.
## Run Steps 1 to 4 for each record which is for python code. Then over all the records take acreage of the accruacy of the C++ to python. This is the accuracy of the LROA fine tuned model.